Layer 2: Task-Importance

**Limited GenAI applicability** — GenAI cannot materially perform the task’s core action.

**Augmentation-oriented** — GenAI can perform part of the task, but substantive human contribution remains necessary.

**Automation-oriented** — GenAI can perform most of the task’s core work.

**Indeterminate** — insufficient information to classify without speculation.

Layer - 2

In [ ]:
# ------------------------------------------------------------
# Step 1 — Load the Layer 2 task universe
#
# Purpose:
# Use the same production task base as Layer 1, retain only the
# three substantive mechanisms, and inspect whether effective
# importance is available before constructing salience shares.
# ------------------------------------------------------------

from pathlib import Path
import pandas as pd

DATA_DIR = Path(
    "/content/drive/MyDrive/"
    "GenAI-task-affordance-labor-outcomes-full"
)

INPUT_FILE = (
    DATA_DIR
    / "02_Task_Data_and_Classification"
    / "Output"
    / "Occupational_Task_Classification_Provisional_v6.csv"
)

tasks = pd.read_csv(INPUT_FILE)

MECHANISM_LABELS = [
    "Automation-oriented",
    "Augmentation-oriented",
    "Limited GenAI applicability",
]

layer2_tasks = tasks[
    tasks["provisional_label"].isin(MECHANISM_LABELS)
].copy()

# Summarize the Layer 2 task universe and importance availability.

layer2_summary = pd.Series({
    "Production task records": len(tasks),
    "Layer 2 mechanism tasks": len(layer2_tasks),
    "SOC6 occupations": layer2_tasks["soc6"].nunique(),
    "Observed importance": layer2_tasks["task_importance"].notna().sum(),
    "Imputed importance": (
        layer2_tasks["task_importance"].isna()
        & layer2_tasks["task_importance_effective"].notna()
    ).sum(),
    "Unavailable importance": (
        layer2_tasks["task_importance_effective"].isna()
    ).sum(),
})

print(layer2_summary)

Purpose: give every substantive task its Core/Supplemental aggregation weight. No importance values are used or modified.

In [ ]:
# ------------------------------------------------------------
# Step 2 — Validate and assign Core/Supplemental task weights
# # Assign weight 2 to Core tasks and weight 1 to
# Supplemental or unrated tasks.
# ------------------------------------------------------------

layer2_tasks["core_weight"] = (
    layer2_tasks["task_type"]
    .map({"Core": 2, "Supplemental": 1})
    .fillna(1) .astype(int)
    )

assert layer2_tasks["core_weight"].isin([1, 2]).all()

print(layer2_tasks["core_weight"].value_counts().sort_index())

Purpose:

calculate the total Core/Supplemental weight carried by each mechanism within every SOC6.

In [ ]:
layer2_weights = (
    layer2_tasks.pivot_table(
        index="soc6",
        columns="provisional_label",
        values="core_weight",
        aggfunc="sum",
        fill_value=0,
    )
    .rename(columns={
        "Automation-oriented": "automation_weight",
        "Augmentation-oriented": "augmentation_weight",
        "Limited GenAI applicability": "limited_weight",
    })
    .reset_index()
)

In [ ]:
# ------------------------------------------------------------
# Step 5 — Calculate Layer 2 mechanism salience shares
# ------------------------------------------------------------

layer2_profiles = layer2_weights.copy()

weight_cols = [
    "automation_weight",
    "augmentation_weight",
    "limited_weight",
]

layer2_profiles["total_weight"] = layer2_profiles[weight_cols].sum(axis=1)

layer2_profiles[
    [
        "automation_core_weighted_share",
        "augmentation_core_weighted_share",
        "limited_core_weighted_share",
    ]
] = layer2_profiles[weight_cols].div(
    layer2_profiles["total_weight"], axis=0
)

In [ ]:
# ------------------------------------------------------------
# Step 7 — Save Layer 2 output
# ------------------------------------------------------------

OUTPUT_FILE = (
    DATA_DIR
    / "04_Layer2_Task_Affordance_Weighted"
    / "Output"
    / "Occupational_Task_Affordance_Layer2_Salience_v6.csv"
)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

layer2_profiles.to_csv(OUTPUT_FILE, index=False)

print(f"Saved: {OUTPUT_FILE}")